# Large Run Step 4: Spatial Statistics Driver

Purpose: run spatial-statistics preparation and figures from already materialized metric outputs, keeping expensive table preparation out of the notebook.

Outputs: metric field tables, station bias summaries, Moran/distance correlation summaries, clustering/PCA/geology summary tables, and spatial figures when requested.


## Setup
Purpose: load config/output paths and driver settings.

Outputs: printed paths and helper variables.


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import textwrap
import time

import pandas as pd

from spatial_vtk.config import SpatialVTKConfig
from spatial_vtk.config.outputs import resolve_output_path
from spatial_vtk.io import load_output_table, preview_output_table, write_output_table


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "spatial_vtk").exists():
            return candidate
    return start


repo_root = find_repo_root()
config_candidates = [
    repo_root / "runs" / "spatial_vtk_config.yaml",
    Path.cwd() / "spatial_vtk_config.yaml",
    Path.cwd() / "runs" / "spatial_vtk_config.yaml",
]
config_path = next((path for path in config_candidates if path.exists()), config_candidates[0])
cfg = SpatialVTKConfig.from_file(config_path).activate()

outputs_root = Path(cfg.path("outputs.root", create_parent=True) or (cfg.root_dir / "runs" / "outputs"))
tables_dir = Path(cfg.path("outputs.tables", create_parent=True) or (outputs_root / "tables"))
figures_dir = Path(cfg.path("outputs.figures", create_parent=True) or (outputs_root / "figures"))
slurm_dir = outputs_root / "slurm"
logs_dir = outputs_root / "logs"
for directory in (tables_dir, figures_dir, slurm_dir, logs_dir):
    directory.mkdir(parents=True, exist_ok=True)

RUN_LOCAL = os.environ.get("SVTK_RUN_LOCAL", "0") == "1"
SUBMIT_SLURM = os.environ.get("SVTK_SUBMIT_SLURM", "0") == "1"
OVERWRITE = os.environ.get("SVTK_OVERWRITE", "0") == "1"
QC_CHUNKSIZE = int(os.environ.get("SVTK_QC_CHUNKSIZE", "1000000"))
PREVIEW_ROWS = int(os.environ.get("SVTK_PREVIEW_ROWS", "5"))

print(f"repo_root: {repo_root}")
print(f"config_path: {config_path}")
print(f"outputs_root: {outputs_root}")
print(f"tables_dir: {tables_dir}")
print(f"figures_dir: {figures_dir}")
print(f"SUBMIT_SLURM={SUBMIT_SLURM} RUN_LOCAL={RUN_LOCAL} OVERWRITE={OVERWRITE}")


## Helpers
Purpose: provide skip/status/Slurm helpers.

Outputs: helper functions only.


In [ ]:
def exists(path):
    return Path(path).exists()


def should_run(*paths):
    return OVERWRITE or not all(Path(path).exists() for path in paths)


def file_info(path):
    path = Path(path)
    if not path.exists():
        return {"path": str(path), "exists": False, "size_gb": None, "modified": None}
    return {
        "path": str(path),
        "exists": True,
        "size_gb": round(path.stat().st_size / 1024**3, 3),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }


def show_files(paths):
    display(pd.DataFrame([file_info(path) for path in paths]))


def submit_script(script_path, *, submit=SUBMIT_SLURM):
    script_path = Path(script_path)
    print(f"script: {script_path}")
    if submit:
        result = subprocess.run(["sbatch", str(script_path)], text=True, capture_output=True, check=False)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"sbatch failed with return code {result.returncode}")
    else:
        print(f"Set SVTK_SUBMIT_SLURM=1 and rerun this cell, or submit manually:")
        print(f"sbatch {shlex.quote(str(script_path))}")


def write_python_slurm_script(script_name, python_body, *, job_name, walltime="24:00:00", memory="32G", cpus=1):
    script_path = slurm_dir / script_name
    body = textwrap.dedent(python_body).strip()
    script = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --output={logs_dir}/{job_name}_%j.out
#SBATCH --error={logs_dir}/{job_name}_%j.err
#SBATCH --time={walltime}
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}

set -euo pipefail
cd {repo_root}
source /project2/jvidale_1700/miniforge3/etc/profile.d/conda.sh || true
conda activate spatial-vtk-py312 || true
python - <<'PYJOB'
{body}
PYJOB
"""
    script_path.write_text(script, encoding="utf-8")
    script_path.chmod(0o755)
    return script_path


def preview_table_path(path, n=PREVIEW_ROWS, columns=None):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() in {".parquet", ".pq"}:
        import pyarrow.parquet as pq
        parquet = pq.ParquetFile(path)
        available = parquet.schema.names
        selected = [column for column in (columns or available) if column in available]
        for batch in parquet.iter_batches(batch_size=max(int(n), 1), columns=selected):
            return batch.to_pandas().head(n)
        return pd.DataFrame(columns=selected)
    return pd.read_csv(path, nrows=n, usecols=columns)


## Resolve Spatial Inputs and Outputs
Purpose: show which metric inputs and spatial outputs already exist.

Outputs: status table only.


In [ ]:
metrics_long_path = resolve_output_path("metrics_long", kind="table", create_parent=True)
metric_field_path = resolve_output_path("metric_field", kind="table", create_parent=True)
event_centered_path = resolve_output_path("event_centered_residuals", kind="table", create_parent=True)
station_bias_path = resolve_output_path("station_bias", kind="table", create_parent=True)
morans_i_path = resolve_output_path("morans_i", kind="table", create_parent=True)
distance_corr_path = resolve_output_path("distance_bin_correlations", kind="table", create_parent=True)
clusters_path = resolve_output_path("clusters", kind="table", create_parent=True)
pca_scores_path = resolve_output_path("pca_station_scores", kind="table", create_parent=True)
geology_path = resolve_output_path("geology_contrasts", kind="table", create_parent=True)

show_files([metrics_long_path, metric_field_path, event_centered_path, station_bias_path, morans_i_path, distance_corr_path, clusters_path, pca_scores_path, geology_path])


## Submit Spatial Summary Build
Purpose: run spatial summary construction as a batch script. The notebook only submits and later previews output files.

Outputs: standard spatial-statistics tables when metric outputs are ready.


In [ ]:
if not metrics_long_path.exists():
    print("metrics_long.parquet is not ready yet; finish Step 3 first.")
elif should_run(metric_field_path, event_centered_path, station_bias_path):
    script = write_python_slurm_script(
        "step04_spatial_summaries.slurm",
        f"""
        import pandas as pd
        from spatial_vtk.config import SpatialVTKConfig
        from spatial_vtk.config.outputs import resolve_output_path
        from spatial_vtk.io import write_output_table
        from spatial_vtk.spatial.calculate import build_metric_field, center_field_by_event, summarize_station_bias

        cfg = SpatialVTKConfig.from_file({str(config_path)!r}).activate()
        metrics = pd.read_parquet(resolve_output_path("metrics_long", kind="table", cfg=cfg, create_parent=True))
        # Use all rows available in metrics_long; filter in config/upstream if you need a narrower run.
        metric_field = build_metric_field(metrics, metric="PGA") if "metric" in metrics.columns else metrics.copy()
        write_output_table("metric_field", metric_field)
        centered = center_field_by_event(metric_field) if "event_id" in metric_field.columns else metric_field.copy()
        write_output_table("event_centered_residuals", centered)
        station_bias = summarize_station_bias(centered) if "station" in centered.columns else centered.head(0)
        write_output_table("station_bias", station_bias)
        print(f"metric_field={{len(metric_field)}} centered={{len(centered)}} station_bias={{len(station_bias)}}")
        """,
        job_name="svtk-step04-spatial",
        walltime="12:00:00",
        memory="64G",
        cpus=1,
    )
    submit_script(script)
else:
    print("Core spatial summary tables already exist; skipping.")


## Render Spatial Figures
Purpose: render station-bias and related maps from compact spatial summary tables only.

Outputs: Step 4 spatial figures when `SVTK_MAKE_FIGURES=1`.


In [ ]:
MAKE_FIGURES = os.environ.get("SVTK_MAKE_FIGURES", "0") == "1"
if not MAKE_FIGURES:
    print("Skipping figures. Set SVTK_MAKE_FIGURES=1 to render them.")
elif not station_bias_path.exists():
    print("station_bias table is not ready yet.")
else:
    from spatial_vtk.spatial.map import plot_station_bias_map
    station_bias = load_output_table("station_bias")
    plot_station_bias_map(station_bias, showfig=False, savefig=True)
    print("Wrote spatial figures from compact summaries.")


## Preview Spatial Summaries
Purpose: inspect small samples only.

Outputs: bounded previews of compact spatial tables.


In [ ]:
for key in ["metric_field", "event_centered_residuals", "station_bias"]:
    path = resolve_output_path(key, kind="table", create_parent=True)
    print(f"\n{key}: {path}")
    if path.exists():
        display(preview_output_table(key, nrows=PREVIEW_ROWS))
